In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import joblib
import os
import config
from itertools import combinations

def compare_train_val_test(train_df, val_df, test_df, clinical_cols, save_table=False, table_fn=''):
    """
    Compare clinical variable distributions across training, validation, and test sets.
    Performs pairwise tests and reports significant differences.

    Parameters
    ----------
    train_df, val_df, test_df : pd.DataFrame
        Datasets to compare.
    clinical_cols : list of str
        Columns to include in the comparison.
    save_table : bool, optional
        If True, save the output to a CSV.
    table_fn : str, optional
        Filename to save table to if save_table=True.

    Returns
    -------
    pd.DataFrame
        Summary table comparing all sets.
    """
    sets = {'Train': train_df, 'Validation': val_df, 'Test': test_df}
    pair_names = list(combinations(sets.keys(), 2))  # [('Train','Validation'), ('Train','Test'), ('Validation','Test')]
    
    # Separate categorical vs numerical
    categorical_cols = [c for c in clinical_cols if train_df[c].dtype == 'object' or train_df[c].dtype.name == 'category']
    numerical_cols = [c for c in clinical_cols if c not in categorical_cols]

    all_results = []

    # ---------- CATEGORICAL ----------
    for col in categorical_cols:
        combined = pd.concat([
            train_df[[col]].assign(Source='Train'),
            val_df[[col]].assign(Source='Validation'),
            test_df[[col]].assign(Source='Test')
        ])
        combined[col] = combined[col].fillna('NAN')
        
        for (s1, s2) in pair_names:
            subset = combined[combined['Source'].isin([s1, s2])]
            tab = pd.crosstab(subset[col], subset['Source'])
            
            if tab.shape == (2, 2):
                _, pval = stats.fisher_exact(tab)
            else:
                try:
                    _, pval, _, _ = stats.chi2_contingency(tab)
                except Exception:
                    pval = np.nan
            
            sig = "YES" if (pval < 0.05) else "NO"
            all_results.append([col, 'categorical', s1, s2, f"{pval:.3g}", sig])

    # ---------- NUMERICAL ----------
    for col in numerical_cols:
        for (s1, s2) in pair_names:
            vals1 = sets[s1][col].dropna()
            vals2 = sets[s2][col].dropna()
            if len(vals1) == 0 or len(vals2) == 0:
                continue

            try:
                _, pval = stats.mannwhitneyu(vals1, vals2, alternative='two-sided')
            except Exception:
                pval = np.nan

            sig = "YES" if (pval < 0.05) else "NO"
            all_results.append([col, 'numeric', s1, s2, f"{pval:.3g}", sig])

    # ---------- COMBINE ----------
    out = pd.DataFrame(all_results, columns=['Variable', 'Type', 'Set 1', 'Set 2', 'p-value', 'Significant (<0.05)'])
    
    if save_table:
        out.to_csv(table_fn or 'train_val_test_comparison.csv', index=False)

    # Print summary of significant differences
    sig_rows = out[out['Significant (<0.05)'] == 'YES']
    if len(sig_rows) > 0:
        print("\nStatistically significant differences detected:")
        for _, row in sig_rows.iterrows():
            print(f" - {row['Variable']} ({row['Type']}): {row['Set 1']} vs {row['Set 2']} (p = {row['p-value']})")
    else:
        print("\nNo statistically significant differences found.")

    return out


# ---------- Example usage ----------
X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_val.joblib"))
y_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_val.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))

summary = compare_train_val_test(
    X_train, X_val, X_test, clinical_cols,
    save_table=True,
    table_fn='train_val_test_comparison.csv'
)

summary.head(100)



✅ No statistically significant differences found.


,Variable,Type,Set 1,Set 2,p-value,Significant (<0.05)
0,SUBTYPE,categorical,Train,Validation,0.969,NO
1,SUBTYPE,categorical,Train,Test,0.786,NO
2,SUBTYPE,categorical,Validation,Test,0.875,NO
3,CANCER_TYPE_ACRONYM,categorical,Train,Validation,1,NO
4,CANCER_TYPE_ACRONYM,categorical,Train,Test,1,NO
...,...,...,...,...,...,...
91,DFS_MONTHS,numeric,Train,Test,0.135,NO
92,DFS_MONTHS,numeric,Validation,Test,0.116,NO
93,PFS_MONTHS,numeric,Train,Validation,0.333,NO
94,PFS_MONTHS,numeric,Train,Test,0.321,NO


In [3]:
import pandas as pd

def summarize_status_columns(train_df, val_df, test_df, status_cols):
    """
    Summarize the count of positive vs negative samples for each status column across splits.
    
    Parameters
    ----------
    train_df, val_df, test_df : pd.DataFrame
        DataFrames containing the status columns.
    status_cols : list of str
        List of status columns to summarize (e.g. ['OS_STATUS', 'DSS_STATUS', 'DFS_STATUS']).

    Returns
    -------
    pd.DataFrame
        A summary table with counts and percentages per split.
    """
    splits = {'Train': train_df, 'Validation': val_df, 'Test': test_df}
    results = []

    for col in status_cols:
        for split_name, df in splits.items():
            if col not in df.columns:
                continue
            
            counts = df[col].value_counts(dropna=False)
            total = counts.sum()
            for status_value, count in counts.items():
                percent = 100 * count / total if total > 0 else 0
                results.append({
                    'Variable': col,
                    'Split': split_name,
                    'Value': status_value,
                    'Count': int(count),
                    'Percent': f"{percent:.1f}%"
                })

    summary = pd.DataFrame(results)
    summary = summary.sort_values(by=['Variable', 'Split']).reset_index(drop=True)

    # Print in a friendly way
    print("\nOutcome Balance Summary (Positive vs Negative by Split):\n")
    for col in status_cols:
        print(f"--- {col} ---")
        display_df = summary[summary['Variable'] == col][['Split', 'Value', 'Count', 'Percent']]
        print(display_df.to_string(index=False))
        print()

    return summary

status_cols = ['OS_STATUS', 'DSS_STATUS', 'DFS_STATUS']
summary = summarize_status_columns(X_train, X_val, X_test, status_cols)

print(sum(y_train), sum(y_val), sum(y_test))


Outcome Balance Summary (Positive vs Negative by Split):

--- OS_STATUS ---
     Split      Value  Count Percent
      Test   0:LIVING     54   79.4%
      Test 1:DECEASED     14   20.6%
     Train   0:LIVING    263   83.2%
     Train 1:DECEASED     53   16.8%
Validation   0:LIVING     64   94.1%
Validation 1:DECEASED      4    5.9%

--- DSS_STATUS ---
     Split                      Value  Count Percent
      Test 0:ALIVE OR DEAD TUMOR FREE     59   86.8%
      Test          1:DEAD WITH TUMOR      9   13.2%
     Train 0:ALIVE OR DEAD TUMOR FREE    278   88.0%
     Train          1:DEAD WITH TUMOR     36   11.4%
     Train                        NaN      2    0.6%
Validation 0:ALIVE OR DEAD TUMOR FREE     67   98.5%
Validation          1:DEAD WITH TUMOR      1    1.5%

--- DFS_STATUS ---
     Split                 Value  Count Percent
      Test         0:DiseaseFree     47   69.1%
      Test                   NaN     17   25.0%
      Test 1:Recurred/Progressed      4    5.9%
     Tra

In [4]:
import pickle
import json

# Load results
with open("gridsearch_results_summary.pkl", "rb") as f:
    grid_results = pickle.load(f)

# --- Option 1: pretty-print as readable JSON ---
print(json.dumps(grid_results, indent=4, default=str))



{
    "{'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_n_boots': 50, 'mrna_fpr_alpha': 0.01, 'mrna_stability_threshold': 0.65, 'mut_n_boots': 50, 'mut_fpr_alpha': 0.01, 'mut_stability_threshold': 0.65}": {
        "internal": {
            "auroc": [
                0.5708792329661362,
                0.06934596117161364
            ],
            "auprc": [
                0.25881005425291403,
                0.07553217106015273
            ],
            "precision": [
                0.337037037037037,
                0.09318330092895646
            ],
            "recall": [
                0.46228070175438596,
                0.2266266734583708
            ],
            "f1": [
                0.3497835497835498,
                0.058388624415887
            ]
        },
        "external": {
            "auroc": [
                0.7020979020979021,
                0.07279448212404534
            ],
            "auprc": [
                0.3482928899615447,
                0

In [7]:
import pickle
import ast

with open("gridsearch_results_summary.pkl", "rb") as f:
    results_summary = pickle.load(f)

for i, (params, metrics) in enumerate(results_summary.items()):
    # Handle string keys
    if isinstance(params, str):
        try:
            params_dict = ast.literal_eval(params)
        except Exception:
            params_dict = {"params": params}
    else:
        params_dict = params

    print("=" * 80)
    print(f"Configuration {i+1}")
    print("Hyperparameters:")
    for k, v in params_dict.items():
        print(f"  {k}: {v}")

    print("\nPerformance Metrics:")
    for dataset_type, dataset_metrics in metrics.items():
        print(f"  {dataset_type.capitalize()}:")
        for metric_name, values in dataset_metrics.items():
            if isinstance(values, (list, tuple)) and len(values) == 2:
                mean, std = values
                print(f"    {metric_name:<15}: {mean:.4f} ± {std:.4f}")
            else:
                print(f"    {metric_name:<15}: {values}")
        print()


Configuration 1
Hyperparameters:
  dropout: 0
  hidden_dim: 32
  lr: 0.001
  mrna_n_boots: 50
  mrna_fpr_alpha: 0.01
  mrna_stability_threshold: 0.65
  mut_n_boots: 50
  mut_fpr_alpha: 0.01
  mut_stability_threshold: 0.65

Performance Metrics:
  Internal:
    auroc          : 0.5709 ± 0.0693
    auprc          : 0.2588 ± 0.0755
    precision      : 0.3370 ± 0.0932
    recall         : 0.4623 ± 0.2266
    f1             : 0.3498 ± 0.0584

  External:
    auroc          : 0.7021 ± 0.0728
    auprc          : 0.3483 ± 0.0527
    precision      : 0.2903 ± 0.0558
    recall         : 0.4359 ± 0.2350
    f1             : 0.3290 ± 0.0865

Configuration 2
Hyperparameters:
  dropout: 0
  hidden_dim: 32
  lr: 0.001
  mrna_n_boots: 50
  mrna_fpr_alpha: 0.01
  mrna_stability_threshold: 0.65
  mut_n_boots: 50
  mut_fpr_alpha: 0.01
  mut_stability_threshold: 0.75

Performance Metrics:
  Internal:
    auroc          : 0.5784 ± 0.0924
    auprc          : 0.2724 ± 0.0946
    precision      : 0.3315 ± 

In [8]:
import pickle
import ast

# === Load results ===
with open("gridsearch_results_summary.pkl", "rb") as f:
    results_summary = pickle.load(f)

# To store best hyperparams for each metric (internal/external)
best_internal = {}
best_external = {}

print("\nDETAILED GRID SEARCH RESULTS")
print("=" * 80)

for i, (params, metrics) in enumerate(results_summary.items()):
    # Parse hyperparameters
    if isinstance(params, str):
        try:
            params_dict = ast.literal_eval(params)
        except Exception:
            params_dict = {"params": params}
    else:
        params_dict = params

    print("=" * 80)
    print(f"Configuration {i+1}")
    print("Hyperparameters:")
    for k, v in params_dict.items():
        print(f"  {k}: {v}")

    print("\nPerformance Metrics:")
    for dataset_type, dataset_metrics in metrics.items():
        print(f"  {dataset_type.capitalize()}:")
        for metric_name, values in dataset_metrics.items():
            if isinstance(values, (list, tuple)) and len(values) == 2:
                mean, std = values
                print(f"    {metric_name:<15}: {mean:.4f} ± {std:.4f}")

                # Track best hyperparams per metric
                if dataset_type == "internal":
                    if metric_name not in best_internal or mean > best_internal[metric_name]["mean"]:
                        best_internal[metric_name] = {
                            "mean": mean,
                            "std": std,
                            "params": params_dict
                        }
                elif dataset_type == "external":
                    if metric_name not in best_external or mean > best_external[metric_name]["mean"]:
                        best_external[metric_name] = {
                            "mean": mean,
                            "std": std,
                            "params": params_dict
                        }
            else:
                print(f"    {metric_name:<15}: {values}")
        print()

# === Print summary of best configs ===
print("\n" + "=" * 80)
print("BEST CONFIGURATIONS SUMMARY")
print("=" * 80)

def print_best_section(title, best_dict):
    print(f"\n{title}")
    for metric, info in best_dict.items():
        print(f"  {metric:<12}: {info['mean']:.4f} ± {info['std']:.4f}")
        print("    Best hyperparameters:")
        for k, v in info["params"].items():
            print(f"      {k}: {v}")
        print()

print_best_section("Internal Metrics", best_internal)
print_best_section("External Metrics", best_external)



DETAILED GRID SEARCH RESULTS
Configuration 1
Hyperparameters:
  dropout: 0
  hidden_dim: 32
  lr: 0.001
  mrna_n_boots: 50
  mrna_fpr_alpha: 0.01
  mrna_stability_threshold: 0.65
  mut_n_boots: 50
  mut_fpr_alpha: 0.01
  mut_stability_threshold: 0.65

Performance Metrics:
  Internal:
    auroc          : 0.5709 ± 0.0693
    auprc          : 0.2588 ± 0.0755
    precision      : 0.3370 ± 0.0932
    recall         : 0.4623 ± 0.2266
    f1             : 0.3498 ± 0.0584

  External:
    auroc          : 0.7021 ± 0.0728
    auprc          : 0.3483 ± 0.0527
    precision      : 0.2903 ± 0.0558
    recall         : 0.4359 ± 0.2350
    f1             : 0.3290 ± 0.0865

Configuration 2
Hyperparameters:
  dropout: 0
  hidden_dim: 32
  lr: 0.001
  mrna_n_boots: 50
  mrna_fpr_alpha: 0.01
  mrna_stability_threshold: 0.65
  mut_n_boots: 50
  mut_fpr_alpha: 0.01
  mut_stability_threshold: 0.75

Performance Metrics:
  Internal:
    auroc          : 0.5784 ± 0.0924
    auprc          : 0.2724 ± 0.0946


In [31]:
print("STARTING CELL")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import os
from preprocessing_utils import *

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests


import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency, mannwhitneyu

def stratified_split_with_balance_check(
    df, y, clinical_cols, test_size=0.15, val_size=0.15,
    max_attempts=100, p_thresh=0.01
):
    """
    Split the dataset into train/val/test (70/15/15) stratified by y,
    and ensure clinical features are balanced across splits.

    A split is rejected if ANY clinical feature has p < p_thresh.
    """

    for attempt in range(max_attempts):
        print("on attempt", attempt)
        
        # Step 1: split into train+val and test
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            df, y, test_size=test_size, stratify=y, random_state=attempt
        )

        # Step 2: split trainval into train and val
        rel_val_size = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval, test_size=rel_val_size,
            stratify=y_trainval, random_state=attempt
        )

        # Step 3: test for imbalance across splits for clinical features
        p_values = []
        reject_split = False

        for feature in clinical_cols:
            print("feature:", feature)

            # Handle numeric features
            if np.issubdtype(df[feature].dtype, np.number):
                vals = [
                    X_train[feature].dropna(),
                    X_val[feature].dropna(),
                    X_test[feature].dropna()
                ]
                pairs = [(0, 1), (0, 2), (1, 2)]
                for (i, j) in pairs:
                    if len(vals[i]) > 0 and len(vals[j]) > 0:
                        if vals[i].nunique() > 1 or vals[j].nunique() > 1:
                            try:
                                _, p = mannwhitneyu(vals[i], vals[j], alternative='two-sided')
                                print(f"  numerical ({i}-{j}) p =", p)
                                p_values.append(p)
                                if p < p_thresh:
                                    reject_split = True
                            except Exception as e:
                                print(f"  numerical test failed ({i}-{j}):", e)

            # Handle categorical features
            else:
                tmp = pd.concat([
                    X_train.assign(split='train'),
                    X_val.assign(split='val'),
                    X_test.assign(split='test')
                ])
                contingency = pd.crosstab(tmp[feature], tmp['split'])
                if contingency.shape[0] > 1 and contingency.shape[1] > 1:
                    try:
                        _, p, _, _ = chi2_contingency(contingency)
                        print(f"  categorical p =", p)
                        p_values.append(p)
                        if p < p_thresh:
                            reject_split = True
                    except Exception as e:
                        print("  categorical test failed:", e)
                else:
                    print("  categorical skipped (not enough variation)")

        # Step 4: if any p < threshold, reject split
        if not reject_split:
            print(f"Balanced split achieved after {attempt+1} attempts.")
            return X_train, X_val, X_test, y_train, y_val, y_test
        else:
            print(f"Rejected split {attempt+1} due to imbalance (min p = {min(p_values):.4g})")

    print("Could not achieve balance after max attempts.")
    return X_train, X_val, X_test, y_train, y_val, y_test


# Load split data
X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_val.joblib"))
y_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_val.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))

# Combine all splits back into one DataFrame
X_full = pd.concat([X_train, X_val, X_test], axis=0)
y_full = pd.concat([y_train, y_val, y_test], axis=0)

# Ensure alignment
X_full = X_full.reset_index(drop=True)
y_full = y_full.reset_index(drop=True)

CLIN_COLS_TO_STRATIFY_ON = ['SUBTYPE', 'AGE', 'RACE', 'ETHNICITY', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'PATH_T_STAGE', 'PATH_N_STAGE', 'PATH_M_STAGE', 'OS_STATUS', 'DSS_STATUS', 'DFS_STATUS', 'PFS_STATUS']

# clin_cols_to_stratify = ['SUBTYPE', 'OS_STATUS', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'AGE']

X_train, X_val, X_test, y_train, y_val, y_test = stratified_split_with_balance_check(
    df=X_full,
    y=y_full,
    clinical_cols=clin_cols_to_stratify,
    test_size=0.15,
    val_size=0.15,
    max_attempts=100,
    p_thresh=0.05
)



STARTING CELL
on attempt 0
feature: SUBTYPE
  categorical p = 0.04773321473697025
feature: AGE
  numerical (0-1) p = 0.9376898865943928
  numerical (0-2) p = 0.9088162393113718
  numerical (1-2) p = 0.8073155556413746
feature: RACE
  categorical p = 0.8478605902736228
feature: ETHNICITY
  categorical p = 0.9431368204109817
feature: ICD_10
  categorical p = 0.08787628678285493
feature: ICD_O_3_HISTOLOGY
  categorical p = 0.16675957517137413
feature: NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT
  categorical p = 0.9882091395571038
feature: PATH_T_STAGE
feature: PATH_N_STAGE
feature: PATH_M_STAGE
feature: OS_STATUS
  categorical p = 0.5710881993773077
feature: DSS_STATUS
  categorical p = 0.9168660486958509
feature: DFS_STATUS
  categorical p = 0.8936308295045327
feature: PFS_STATUS
  categorical p = 0.9179674815626137
Rejected split 1 due to imbalance (min p = 0.04773)
on attempt 1
feature: SUBTYPE
  categorical p = 0.9716070134167116
feature: AGE
  numerical (0-1) p = 0.2079471306676688
  nu

In [4]:
print(clinical_cols)
['SUBTYPE', 'AGE', 'RACE', 'ETHNICITY', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'PATH_T_STAGE', 'PATH_N_STAGE', 'PATH_M_STAGE', 'OS_STATUS', 'DSS_STATUS' 'DFS_STATUS', 'PFS_STATUS']

['SUBTYPE', 'CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID', 'AGE', 'SEX', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION', 'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'ETHNICITY', 'FORM_COMPLETION_DATE', 'HISTORY_NEOADJUVANT_TRTYN', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'INFORMED_CONSENT_VERIFIED', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'PATH_M_STAGE', 'PATH_N_STAGE', 'PATH_T_STAGE', 'PERSON_NEOPLASM_CANCER_STATUS', 'PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT', 'PRIOR_DX', 'RACE', 'RADIATION_THERAPY', 'WEIGHT', 'IN_PANCANPATHWAYS_FREEZE', 'OS_STATUS', 'OS_MONTHS', 'DSS_STATUS', 'DSS_MONTHS', 'DFS_STATUS', 'DFS_MONTHS', 'PFS_STATUS', 'PFS_MONTHS', 'GENETIC_ANCESTRY_LABEL']
